In [1]:
from process_eeg import get_psd_df, get_psd_avg_df, get_osc_df
from visualization.vis_eeg import compare_epo_psd, compare_epo_len_evk, compare_band_metric
from utils.gen_utils import get_sids
import numpy as np

%load_ext autoreload
%autoreload 2

/Users/sophiecaroni/epfl_hes/spanav-tbi/Analysis/spectral_utils.py:1: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  import fooof


In [19]:
test = False
show = True
save = True

# 0. Overview of all epoch-types
## PSD

In [ ]:
# Load dataframe with power spectra
psd_df = get_psd_df(test=test, log=True)
psd_df

In [ ]:
# Load dataframe with average power spectra
psd_avg_df = get_psd_avg_df(log=True)
psd_avg_df

In [ ]:
# Average across subjects
# psd_avg_df = psd_avg_df[psd_avg_df['cond'] != 'cTBS']
compare_epo_psd(psd_avg_df, 'epo_type', show=show, save=save, plot_subj='average')

In [ ]:
# For each subject separately  
psd_df_lin = get_psd_df(test=test, log=False)  # need linear psd dfs to compute std across blocks
for sid in get_sids():
    df = psd_df_lin[psd_df_lin['sid'] == sid]
    df['pw_avg'] = np.log10(df['pw_avg'])
    plot_df = df.groupby(['cond', 'epo_type', 'freq'], as_index=False).agg(  # average together blocks of the same condition
            **{
                "sid": ("sid", 'first'),
                "freq": ("freq", 'first'),
                "pw_avg": ('pw_avg', 'mean'),
                "pw_std": ('pw_avg', 'std'),
                "N": ('pw_avg', 'count'),
            }
        )
    compare_epo_psd(plot_df, 'epo_type', show=show, save=save, plot_subj=sid)  # sem_n is number of blocks by condition

## Oscillations

In [ ]:
# Load dataframe with power spectra
osc_df = get_osc_df(test=test)
# osc_df = osc_df[osc_df['cond'] != 'cTBS']
osc_df 

## Power in canonical bands

In [ ]:
compare_band_metric(osc_df, super_col='epo_type', show=show, save=save, metric_name='abs_pw')
compare_band_metric(osc_df, super_col='epo_type', show=show, save=save, metric_name='rel_pw')

In [ ]:
# For each subject separately
for sid in get_sids(test):
    df = osc_df[osc_df['sid'] == sid]
    # if sid == '04':
    #     continue
    compare_band_metric(df, super_col='epo_type', show=show, save=save, metric_name='abs_pw')
    compare_band_metric(df, super_col='epo_type', show=show, save=save, metric_name='rel_pw')

## SNR 
Based on oscillations / background noise

In [ ]:
# Average across patients
compare_band_metric(osc_df, super_col='epo_type', show=show, save=save, metric_name='osc_snr')

In [ ]:
# For each subject separately
for sid in get_sids(test):
    df = osc_df[osc_df['sid'] == sid]
    compare_band_metric(df, super_col='epo_type', show=show, save=save, metric_name='osc_snr')

# 1. Length of ObjPres epochs
Epochs extracted from 3s windows during which an object to recall is presented

In [ ]:
lens = [1] if test else [1, 3]

## Compare PSDs

In [ ]:
psd_by_len_and_epo = compare_objepo_len('psd', lens, test=test, sid=sid)
psd_by_len_and_epo

In [ ]:
compare_epo_psd(psd_by_len_and_epo, 'epo_s', show=show, save=save)

In [ ]:
compare_epo_psd(psd_by_len_and_epo, 'epo_s', show=show, save=save)

In [ ]:
band_pw_by_len_and_epo = compare_objepo_len('band_pw', lens, test=test, sid=sid)

In [ ]:
compare_band_metric(band_pw_by_len_and_epo, super_col='epo_s', show=show, save=save, metric_name='pw')

## Compare evoked response 

In [ ]:
evk_df = compare_objepo_len('evk', lens, test=test, sid=sid)
evk_df

In [ ]:
compare_epo_len_evk(evk_df, show=show, save=save)

## Compare SNR

In [ ]:
snr_by_len_and_epo = compare_objepo_len('snr', lens, test=test, sid=sid)
snr_by_len_and_epo

In [ ]:
snr_by_len_and_epo = compare_objepo_len('osc_snr', lens, test=test, sid=sid)
snr_by_len_and_epo

In [ ]:
compare_band_metric(snr_by_len_and_epo, super_col='epo_s', show=show, save=save, metric_name='osc_snr')